In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASE_DIR = Path.cwd().parents[1]
DATA_DIR = BASE_DIR / "Data"

SMOS_DIR = DATA_DIR / "SMOS"
SCAN_DIR = DATA_DIR / "SCAN" / "processed"
COMBINED_DIR = DATA_DIR / "combined"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("SMOS_DIR exists:", SMOS_DIR.exists())
print("SCAN_DIR exists:", SCAN_DIR.exists())
print("COMBINED_DIR:", COMBINED_DIR)

BASE_DIR: /Users/m.mughees/Desktop/Pi-515-2026
SMOS_DIR exists: True
SCAN_DIR exists: True
COMBINED_DIR: /Users/m.mughees/Desktop/Pi-515-2026/Data/combined


In [3]:
smos_train_df = pd.read_csv(SMOS_DIR / "train.csv")
smos_test_df = pd.read_csv(SMOS_DIR / "test.csv")

scan_train_df = pd.read_csv(SCAN_DIR / "scan_train.csv")
scan_holdout_df = pd.read_csv(SCAN_DIR / "scan_holdout.csv")

print("SMOS train:", smos_train_df.shape)
print("SMOS test:", smos_test_df.shape)
print("SCAN train:", scan_train_df.shape)
print("SCAN holdout:", scan_holdout_df.shape)

SMOS train: (2099831, 26)
SMOS test: (10000, 26)
SCAN train: (4063, 12)
SCAN holdout: (452, 12)


In [4]:
rename_map = {
    "num__latitude": "latitude",
    "num__longitude": "longitude",
    "num__soil_temperature_level_1": "soil_temperature",
    "num__total_precipitation": "total_precipitation",
    "num__runoff": "runoff",
    "num__total_evaporation": "total_evaporation",
    "num__potential_evaporation": "potential_evaporation",
    "num__2m_dewpoint_temperature": "dewpoint_temperature",
    "num__2m_temperature": "air_temperature",
    "num__snow_cover": "snow_cover",
    "num__snow_depth": "snow_depth",
    "num__snowfall": "snowfall",
    "num__snowmelt": "snowmelt",
    "num__year": "year",
    "num__month": "month",
    "num__day": "day",
    "cat__DistrictNa_Central": "district_central",
    "cat__DistrictNa_East Central": "district_east_central",
    "cat__DistrictNa_North Central": "district_north_central",
    "cat__DistrictNa_Northeast": "district_northeast",
    "cat__DistrictNa_Northwest": "district_northwest",
    "cat__DistrictNa_South Central": "district_south_central",
    "cat__DistrictNa_Southeast": "district_southeast",
    "cat__DistrictNa_Southwest": "district_southwest",
    "cat__DistrictNa_West Central": "district_west_central",
    "Soil_Moisture": "soil_moisture",
}

smos_train_df = smos_train_df.rename(columns=rename_map)
smos_test_df = smos_test_df.rename(columns=rename_map)

smos_train_df.head()

,latitude,longitude,soil_temperature,total_precipitation,runoff,total_evaporation,potential_evaporation,dewpoint_temperature,air_temperature,snow_cover,...,district_central,district_east_central,district_north_central,district_northeast,district_northwest,district_south_central,district_southeast,district_southwest,district_west_central,soil_moisture
0,-0.776986,1.426464,-1.369474,-0.367620,0.264376,1.097594,1.129800,-0.989945,-1.215082,0.768468,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.360234
1,-1.216539,0.035936,1.056937,-0.359515,0.063897,-0.850514,-0.224441,0.796616,0.823547,-0.269393,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.213469
2,-1.028159,1.205244,0.334309,-0.336933,-0.268701,-0.342088,0.007690,0.505021,0.467160,-0.269393,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.166913
3,0.478879,-0.912151,-0.674495,-0.368519,-0.352383,1.054375,0.972629,-0.400612,-0.537826,-0.265868,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.199984
4,-0.400226,0.035936,0.985257,-0.078794,0.318792,-1.182331,-0.441741,1.085233,0.926438,-0.269393,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.214700


In [5]:
common_columns = [
    "latitude",
    "longitude",
    "air_temperature",
    "dewpoint_temperature",
    "total_precipitation",
    "year",
    "month",
    "day",
    "soil_temperature",
    "soil_moisture",
]

In [6]:
smos_train_aligned = smos_train_df[common_columns].copy()
smos_test_aligned = smos_test_df[common_columns].copy()

scan_train_aligned = scan_train_df[common_columns].copy()
scan_holdout_aligned = scan_holdout_df[common_columns].copy()

print("SMOS aligned train:", smos_train_aligned.shape)
print("SMOS aligned test:", smos_test_aligned.shape)
print("SCAN aligned train:", scan_train_aligned.shape)
print("SCAN aligned holdout:", scan_holdout_aligned.shape)

SMOS aligned train: (2099831, 10)
SMOS aligned test: (10000, 10)
SCAN aligned train: (4063, 10)
SCAN aligned holdout: (452, 10)


In [7]:
smos_train_aligned["source"] = "SMOS"
smos_test_aligned["source"] = "SMOS"

scan_train_aligned["source"] = "SCAN"
scan_holdout_aligned["source"] = "SCAN"

In [8]:
combined_train_df = pd.concat(
    [smos_train_aligned, scan_train_aligned],
    ignore_index=True
)

print("Combined train shape:", combined_train_df.shape)
combined_train_df.head()

Combined train shape: (2103894, 11)


,latitude,longitude,air_temperature,dewpoint_temperature,total_precipitation,year,month,day,soil_temperature,soil_moisture,source
0,-0.776986,1.426464,-1.215082,-0.989945,-0.367620,-0.314381,-1.292227,-1.323847,-1.369474,0.360234,SMOS
1,-1.216539,0.035936,0.823547,0.796616,-0.359515,0.372430,0.014939,1.733197,1.056937,0.213469,SMOS
2,-1.028159,1.205244,0.467160,0.505021,-0.336933,-1.230129,-0.965435,1.053854,0.334309,0.166913,SMOS
3,0.478879,-0.912151,-0.537826,-0.400612,-0.368519,-1.230129,1.322104,-1.663519,-0.674495,0.199984,SMOS
4,-0.400226,0.035936,0.926438,1.085233,-0.078794,-1.459067,0.014939,-0.191609,0.985257,0.214700,SMOS


In [9]:
print(combined_train_df["source"].value_counts())

source
SMOS    2099831
SCAN       4063
Name: count, dtype: int64


In [10]:
print("Combined train nulls:")
print(combined_train_df.isnull().sum())

print("\nSMOS test nulls:")
print(smos_test_aligned.isnull().sum())

print("\nSCAN holdout nulls:")
print(scan_holdout_aligned.isnull().sum())

Combined train nulls:
latitude                0
longitude               0
air_temperature         0
dewpoint_temperature    0
total_precipitation     0
year                    0
month                   0
day                     0
soil_temperature        0
soil_moisture           0
source                  0
dtype: int64

SMOS test nulls:
latitude                0
longitude               0
air_temperature         0
dewpoint_temperature    0
total_precipitation     0
year                    0
month                   0
day                     0
soil_temperature        0
soil_moisture           0
source                  0
dtype: int64

SCAN holdout nulls:
latitude                0
longitude               0
air_temperature         0
dewpoint_temperature    0
total_precipitation     0
year                    0
month                   0
day                     0
soil_temperature        0
soil_moisture           0
source                  0
dtype: int64


In [11]:
combined_train_df.to_csv(COMBINED_DIR / "combined_train.csv", index=False)
smos_test_aligned.to_csv(COMBINED_DIR / "smos_test_aligned.csv", index=False)
scan_holdout_aligned.to_csv(COMBINED_DIR / "scan_holdout_aligned.csv", index=False)

print("✅ Saved:")
print(COMBINED_DIR / "combined_train.csv")
print(COMBINED_DIR / "smos_test_aligned.csv")
print(COMBINED_DIR / "scan_holdout_aligned.csv")

✅ Saved:
/Users/m.mughees/Desktop/Pi-515-2026/Data/combined/combined_train.csv
/Users/m.mughees/Desktop/Pi-515-2026/Data/combined/smos_test_aligned.csv
/Users/m.mughees/Desktop/Pi-515-2026/Data/combined/scan_holdout_aligned.csv


In [12]:
print(combined_train_df.columns.tolist())
combined_train_df.head()

['latitude', 'longitude', 'air_temperature', 'dewpoint_temperature', 'total_precipitation', 'year', 'month', 'day', 'soil_temperature', 'soil_moisture', 'source']


,latitude,longitude,air_temperature,dewpoint_temperature,total_precipitation,year,month,day,soil_temperature,soil_moisture,source
0,-0.776986,1.426464,-1.215082,-0.989945,-0.367620,-0.314381,-1.292227,-1.323847,-1.369474,0.360234,SMOS
1,-1.216539,0.035936,0.823547,0.796616,-0.359515,0.372430,0.014939,1.733197,1.056937,0.213469,SMOS
2,-1.028159,1.205244,0.467160,0.505021,-0.336933,-1.230129,-0.965435,1.053854,0.334309,0.166913,SMOS
3,0.478879,-0.912151,-0.537826,-0.400612,-0.368519,-1.230129,1.322104,-1.663519,-0.674495,0.199984,SMOS
4,-0.400226,0.035936,0.926438,1.085233,-0.078794,-1.459067,0.014939,-0.191609,0.985257,0.214700,SMOS
